# Trading platform SQL practice

Run the **setup cell** once. It builds a small fake trading database (`trading_practice.db`) next to this notebook: about 3,000 users, 16,500 trades, 20 assets.

Each task has a collapsed **hint** with the 7 answers below. Click it only after you've tried.

Then write your SQL inside `q(""" ... """)` in the task cells. `q()` runs the query and shows the result as a table.

**Tables**
- `users(user_id, country, signup_date)`
- `trades(trade_id, user_id, asset_id, traded_at, amount_eur)`
- `assets(asset_id, symbol, asset_class)`, asset_class is 'crypto', 'stock', 'etf' or 'metal'

**The 7 questions before every query**
1. What does one row of the result look like?
2. Which tables hold the columns I need?
3. Must one table stay complete? (LEFT) or only matches? (INNER)
4. Which rows do I throw away first? (WHERE)
5. Do I collapse rows into totals? (GROUP BY)
6. Do I filter on those totals? (HAVING)
7. Order? (ORDER BY)

Fewer rows than the source → GROUP BY. Same rows plus a new column → OVER.

Engine: **SQLite**. Dates are text like `'2026-10-15 13:45:00'`, so `traded_at >= '2026-10-01'` works.

In [1]:
# SETUP: run once (re-running rebuilds the same data)
import sqlite3, random
from datetime import datetime, timedelta
import pandas as pd

random.seed(42)
con = sqlite3.connect("trading_practice.db")
cur = con.cursor()
cur.executescript("""
DROP TABLE IF EXISTS trades; DROP TABLE IF EXISTS users; DROP TABLE IF EXISTS assets;
CREATE TABLE users  (user_id INTEGER PRIMARY KEY, country TEXT, signup_date TEXT);
CREATE TABLE assets (asset_id INTEGER PRIMARY KEY, symbol TEXT, asset_class TEXT);
CREATE TABLE trades (trade_id INTEGER PRIMARY KEY, user_id INTEGER, asset_id INTEGER,
                     traded_at TEXT, amount_eur REAL);
""")

assets = [("BTC","crypto"),("ETH","crypto"),("SOL","crypto"),("XRP","crypto"),("ADA","crypto"),
          ("DOGE","crypto"),("AAPL","stock"),("MSFT","stock"),("TSLA","stock"),("NVDA","stock"),
          ("SAP","stock"),("OMV","stock"),("VWCE","etf"),("IWDA","etf"),("EUNL","etf"),
          ("XAU","metal"),("XAG","metal"),("XPT","metal"),("XPD","metal"),("SPYI","etf")]
cur.executemany("INSERT INTO assets VALUES (?,?,?)",
                [(i+1, s, c) for i, (s, c) in enumerate(assets)])
by_class = {}
for i, (s, c) in enumerate(assets):
    by_class.setdefault(c, []).append(i+1)

countries = ["AT"]*35 + ["DE"]*30 + ["FR"]*10 + ["IT"]*10 + ["ES"]*8 + ["SK"]*7
start = datetime(2024, 1, 1)
users = []
for uid in range(1, 3001):
    signup = start + timedelta(days=random.randint(0, 1040))
    users.append((uid, random.choice(countries), signup.strftime("%Y-%m-%d")))
cur.executemany("INSERT INTO users VALUES (?,?,?)", users)

end = datetime(2026, 11, 10)
trades, tid = [], 1
for uid, country, signup in users:
    if random.random() < 0.18:        # some users never trade
        continue
    s = datetime.strptime(signup, "%Y-%m-%d")
    n = random.choice([1, 1, 2, 3, 5, 8, 13, 20])
    crypto_first = random.random() < 0.65
    for k in range(n):
        if k == 0:
            cls = "crypto" if crypto_first else random.choice(["stock", "etf", "metal"])
        else:
            cls = random.choices(["crypto", "stock", "etf", "metal"], [55, 25, 15, 5])[0]
        ts = s + timedelta(days=random.randint(0, max(1, (end - s).days)),
                           seconds=random.randint(0, 86399))
        if k == 0:
            ts = s + timedelta(days=random.randint(0, 20), seconds=random.randint(0, 86399))
        if ts > end:
            ts = end - timedelta(seconds=random.randint(0, 86399))
        amt = round(random.lognormvariate(5, 1.1), 2)
        trades.append((tid, uid, random.choice(by_class[cls]), ts.strftime("%Y-%m-%d %H:%M:%S"), amt))
        tid += 1
cur.executemany("INSERT INTO trades VALUES (?,?,?,?,?)", trades)
con.commit()

def q(sql):
    """Run a SQL query and show the result."""
    return pd.read_sql_query(sql, con)

print(f"Built: {len(users)} users, {len(trades)} trades, {len(assets)} assets")
q("SELECT * FROM trades LIMIT 5")

Built: 3000 users, 16512 trades, 20 assets


,trade_id,user_id,asset_id,traded_at,amount_eur
0,1,1,3,2024-08-17 06:54:44,77.12
1,2,2,1,2025-08-05 01:59:09,122.22
2,3,2,10,2025-09-12 06:32:42,227.69
3,4,2,2,2025-07-29 06:30:45,63.72
4,5,2,14,2025-09-01 09:06:18,57.70


### Warm-up
Look at each table. How many rows does each have?

*Answer the 7 questions yourself first, then open the hint to check.*

<details>
<summary>💡 Hint: the 7 answers</summary>

<ol>
<li><b>One row:</b> a single row with three counts</li>
<li><b>Tables:</b> trades, users, assets (each counted on its own)</li>
<li><b>Join:</b> none. Three scalar subqueries, <code>(SELECT COUNT(*) FROM …)</code>, side by side</li>
<li><b>WHERE:</b> none</li>
<li><b>GROUP BY:</b> no</li>
<li><b>HAVING:</b> no</li>
<li><b>ORDER BY:</b> no (only one row)</li>
</ol>

</details>

In [2]:
q("""
SELECT
  (SELECT COUNT(*) FROM trades) AS num_trades,
  (SELECT COUNT(*) FROM users)  AS num_users,
  (SELECT COUNT(*) FROM assets) AS num_assets
""")

,num_trades,num_users,num_assets
0,16512,3000,20


### Task 1
Total trading volume (`SUM(amount_eur)`) and number of trades **per country** in **October 2026**, highest volume first.

*Answer the 7 questions yourself first, then open the hint to check.*

<details>
<summary>💡 Hint: the 7 answers</summary>

<ol>
<li><b>One row:</b> one country, with its trade count and volume</li>
<li><b>Tables:</b> trades (amount, date) + users (country)</li>
<li><b>Join:</b> INNER. We only care about users who have trades</li>
<li><b>WHERE:</b> <code>traded_at &gt;= '2026-10-01' AND traded_at &lt; '2026-11-01'</code> (start included, end excluded)</li>
<li><b>GROUP BY:</b> country</li>
<li><b>HAVING:</b> no</li>
<li><b>ORDER BY:</b> volume DESC</li>
</ol>

</details>

In [3]:
q("""
SELECT u.country, COUNT(*) AS num_trades, SUM(t.amount_eur) AS total_volume
FROM trades t
JOIN users u ON u.user_id = t.user_id
WHERE t.traded_at >= '2026-10-01' AND t.traded_at < '2026-11-01'
GROUP BY u.country
ORDER BY total_volume DESC
""")

,country,num_trades,total_volume
0,AT,562,154654.60
1,DE,516,128205.03
2,SK,185,51294.00
3,IT,169,42436.54
4,FR,153,39381.97
5,ES,131,35800.17


### Task 2
Users who **signed up in 2026** but have **never traded**.

*Answer the 7 questions yourself first, then open the hint to check.*

<details>
<summary>💡 Hint: the 7 answers</summary>

<ol>
<li><b>One row:</b> one user (id, signup date)</li>
<li><b>Tables:</b> users + trades</li>
<li><b>Join:</b> LEFT. users must stay complete; non-traders have no match in trades</li>
<li><b>WHERE:</b> signup in 2026 <b>and</b> <code>t.trade_id IS NULL</code> (no match = never traded)</li>
<li><b>GROUP BY:</b> no</li>
<li><b>HAVING:</b> no</li>
<li><b>ORDER BY:</b> optional</li>
</ol>

</details>

In [4]:
q("""
SELECT u.user_id, u.signup_date
FROM users u
LEFT JOIN trades t ON u.user_id = t.user_id
WHERE u.signup_date >= '2026-01-01' AND u.signup_date < '2027-01-01'
  AND t.trade_id IS NULL
""")

,user_id,signup_date
0,36,2026-01-17
1,58,2026-03-28
2,80,2026-11-05
3,93,2026-08-31
4,95,2026-05-16
...,...,...
150,2976,2026-09-03
151,2979,2026-02-04
152,2990,2026-09-30
153,2995,2026-02-18


### Task 3
Each user's **first trade**: user_id, the date, and the **asset class** of that trade.

*Answer the 7 questions yourself first, then open the hint to check.*

<details>
<summary>💡 Hint: the 7 answers</summary>

<ol>
<li><b>One row:</b> one user, with first trade date and asset class</li>
<li><b>Tables:</b> trades + assets (asset_class)</li>
<li><b>Join:</b> INNER. Only users who traded</li>
<li><b>WHERE:</b> <code>rn = 1</code>, after numbering each user's trades by date</li>
<li><b>GROUP BY:</b> no. Same rows plus a new column → <code>ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY traded_at)</code></li>
<li><b>HAVING:</b> no</li>
<li><b>ORDER BY:</b> optional</li>
</ol>

</details>

In [5]:
q("""
WITH ranked AS (
  SELECT t.user_id,
         t.traded_at,
         t.asset_id,
         ROW_NUMBER() OVER (PARTITION BY t.user_id ORDER BY t.traded_at) AS rn
  FROM trades t
)
SELECT r.user_id, r.traded_at AS first_trade_at, a.asset_class
FROM ranked r
JOIN assets a ON a.asset_id = r.asset_id
WHERE r.rn = 1
""")

,user_id,first_trade_at,asset_class
0,1,2024-08-17 06:54:44,crypto
1,2,2025-07-29 06:30:45,crypto
2,4,2024-08-02 12:12:14,crypto
3,5,2024-06-27 22:40:00,crypto
4,6,2026-05-31 10:07:32,crypto
...,...,...,...
2467,2994,2024-03-29 05:29:12,crypto
2468,2996,2025-10-15 03:19:02,crypto
2469,2997,2024-03-28 08:47:14,crypto
2470,2998,2024-08-15 07:09:35,metal


### Task 4 (bonus)
Assets traded by **more than 500 different users**, with the number of users.

*Answer the 7 questions yourself first, then open the hint to check.*

<details>
<summary>💡 Hint: the 7 answers</summary>

<ol>
<li><b>One row:</b> one asset, with its number of distinct users</li>
<li><b>Tables:</b> assets (symbol) + trades (user_id)</li>
<li><b>Join:</b> INNER</li>
<li><b>WHERE:</b> none</li>
<li><b>GROUP BY:</b> asset (symbol)</li>
<li><b>HAVING:</b> <code>COUNT(DISTINCT t.user_id) &gt; 500</code></li>
<li><b>ORDER BY:</b> user count DESC</li>
</ol>

</details>

In [6]:
q("""
SELECT a.symbol, COUNT(DISTINCT t.user_id) AS n_users
FROM assets a
JOIN trades t ON a.asset_id = t.asset_id
GROUP BY a.symbol
HAVING COUNT(DISTINCT t.user_id) > 500   -- repeat the aggregate: aliases in HAVING only work in SQLite/MySQL
ORDER BY n_users DESC
""")

,symbol,n_users
0,BTC,1017
1,ETH,1009
2,XRP,987
3,SOL,985
4,DOGE,982
5,ADA,940
6,OMV,542
7,SAP,520
8,TSLA,518
9,AAPL,512


### Task 5
For each **asset class**, the number of **different users** who traded it in **2026**, but only classes with **more than 1,000** users.

*Answer the 7 questions yourself first, then open the hint to check.*

<details>
<summary>💡 Hint: the 7 answers</summary>

<ol>
<li><b>One row:</b> one asset class, with its number of distinct users</li>
<li><b>Tables:</b> assets (asset_class) + trades</li>
<li><b>Join:</b> INNER</li>
<li><b>WHERE:</b> traded in 2026 (<code>&gt;= '2026-01-01' AND &lt; '2027-01-01'</code>)</li>
<li><b>GROUP BY:</b> asset_class</li>
<li><b>HAVING:</b> <code>COUNT(DISTINCT t.user_id) &gt; 1000</code></li>
<li><b>ORDER BY:</b> user count DESC</li>
</ol>

</details>

In [7]:
q("""
SELECT a.asset_class, COUNT(DISTINCT t.user_id) AS n_users
FROM assets a
JOIN trades t ON a.asset_id = t.asset_id
WHERE t.traded_at >= '2026-01-01' AND t.traded_at < '2027-01-01'
GROUP BY a.asset_class
HAVING COUNT(DISTINCT t.user_id) > 1000
ORDER BY n_users DESC
""")

,asset_class,n_users
0,crypto,1583
1,stock,1106


### Task 6
**Monthly** trading volume in 2026 (one row per month).

*Answer the 7 questions yourself first, then open the hint to check.*

<details>
<summary>💡 Hint: the 7 answers</summary>

<ol>
<li><b>One row:</b> one month, with volume and trade count</li>
<li><b>Tables:</b> trades only</li>
<li><b>Join:</b> none</li>
<li><b>WHERE:</b> traded in 2026</li>
<li><b>GROUP BY:</b> month. <code>substr(traded_at, 1, 7)</code> turns '2026-10-15 13:45:00' into '2026-10'</li>
<li><b>HAVING:</b> no</li>
<li><b>ORDER BY:</b> month</li>
</ol>

</details>

In [8]:
q("""
SELECT substr(t.traded_at, 1, 7) AS month,
       SUM(t.amount_eur)         AS volume_eur,
       COUNT(*)                  AS n_trades
FROM trades t
WHERE t.traded_at >= '2026-01-01' AND t.traded_at < '2027-01-01'
GROUP BY month
ORDER BY month
""")

,month,volume_eur,n_trades
0,2026-01,159501.67,586
1,2026-02,158186.58,563
2,2026-03,191607.12,718
3,2026-04,191571.29,715
4,2026-05,223373.80,819
5,2026-06,231374.30,859
6,2026-07,267131.82,1030
7,2026-08,318944.96,1162
8,2026-09,372454.49,1318
9,2026-10,451772.31,1716


### Task 7 (harder)
Of the users whose **first** trade was **crypto**, how many later bought a **stock**? Give both numbers and the conversion rate. (A typical fintech interview question.)

*Answer the 7 questions yourself first, then open the hint to check.*

<details>
<summary>💡 Hint: the 7 answers</summary>

<ol>
<li><b>One row:</b> a single summary row: crypto-first users, how many of them bought a stock, and the rate in %</li>
<li><b>Tables:</b> trades + assets</li>
<li><b>Join:</b> step 1, INNER trades→assets to rank trades; step 2, LEFT crypto-first users → stock buyers (keep all crypto-first users, count the matches)</li>
<li><b>WHERE:</b> <code>rn = 1 AND asset_class = 'crypto'</code> for the first group; <code>asset_class = 'stock'</code> for the second</li>
<li><b>GROUP BY:</b> no. Aggregates over the whole result give one row</li>
<li><b>HAVING:</b> no</li>
<li><b>ORDER BY:</b> no. <i>Follow-up: why no date check for “later”? Their first trade was crypto, so any stock trade must come after it.</i></li>
</ol>

</details>

In [9]:
q("""
WITH ranked AS (
  SELECT t.user_id, a.asset_class, t.traded_at,
         ROW_NUMBER() OVER (PARTITION BY t.user_id ORDER BY t.traded_at) AS rn
  FROM trades t
  JOIN assets a ON a.asset_id = t.asset_id
),
crypto_first AS (
  SELECT user_id FROM ranked
  WHERE rn = 1 AND asset_class = 'crypto'
),
stock_buyers AS (
  SELECT DISTINCT t.user_id
  FROM trades t
  JOIN assets a ON a.asset_id = t.asset_id
  WHERE a.asset_class = 'stock'
)
SELECT COUNT(*)         AS crypto_first_users,
       COUNT(s.user_id) AS later_bought_stock,
       ROUND(100.0 * COUNT(s.user_id) / COUNT(*), 1) AS conversion_pct   -- 100.0 forces decimal division
FROM crypto_first c
LEFT JOIN stock_buyers s ON s.user_id = c.user_id
""")

,crypto_first_users,later_bought_stock,conversion_pct
0,1554,798,51.4


### Task 8 (extra)
Count trades by size: **small** (under 100 €), **medium** (100 to 999 €), **large** (1,000 € and more), most common first.

*Answer the 7 questions yourself first, then open the hint to check.*

<details>
<summary>💡 Hint: the 7 answers</summary>

<ol>
<li><b>One row:</b> one size bucket, with its trade count</li>
<li><b>Tables:</b> trades only</li>
<li><b>Join:</b> none</li>
<li><b>WHERE:</b> none</li>
<li><b>GROUP BY:</b> the bucket, built with <code>CASE WHEN … THEN … ELSE … END</code>. Conditions are checked top to bottom, first match wins</li>
<li><b>HAVING:</b> no</li>
<li><b>ORDER BY:</b> trade count DESC</li>
</ol>

</details>

In [10]:
q("""
SELECT CASE WHEN t.amount_eur < 100  THEN 'small'
            WHEN t.amount_eur < 1000 THEN 'medium'
            ELSE 'large'
       END AS bucket,
       COUNT(*) AS n_trades
FROM trades t
GROUP BY bucket
ORDER BY n_trades DESC
""")

,bucket,n_trades
0,medium,9886
1,small,5976
2,large,650
